# Figure 1 — Classification of unproductive splice junctions

LeafCutter2 assigns each splice junction to one of three classes using only
start- and stop-codon annotation: **productive**, **unproductive** (the junction
introduces a premature termination codon), or **near-UTR**, where the reading
frame in use cannot be resolved and no call is made.

| Panel | What it shows |
|---|---|
| **c** | Junction tracks at *SRSF4*, coloured by class |
| **d** | Class composition by junction usage quartile |
| **e** | GENCODE v46 transcript-type composition of each class |
| **g** | log2 fold change of junction usage under four perturbations that stabilise NMD substrates |
| **h** | Three rules predicting how efficiently a premature stop triggers NMD |

Of 967,006 junctions, 33% are called unproductive, 29% productive and 38%
near-UTR. The split tracks how often a junction is used: 55% of the most-used
quartile is productive, against 17% of the least-used.

In [ ]:
import os
from matplotlib import pyplot as plt

import Figure1_helpers as H
import Figure1_plot_helpers as P

plt.rcParams['svg.fonttype'] = 'none'
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

PLOTS_DIR = 'plots'
os.makedirs(PLOTS_DIR, exist_ok=True)
print('reading pipeline output from', H.SOURCE)

In [ ]:
# Fast path: reload what run_all() pickled.
data = H.load_plot_data('figure_data')

fig1c_tracks = data['fig1c_tracks']
fig1d_data   = data['fig1d_data']
fig1e_data   = data['fig1e_data']
fig1g_data   = data['fig1g_data']
fig1h_data   = data['fig1h_data']

In [ ]:
# Full rebuild. Reads the 50 MB classification table and the 43 MB log2FC
# cache, so it takes a few minutes. Run once, then use the cell above.

# data = H.run_all('figure_data')

### Fig. 1c — SRSF4 browser tracks

The published panel is an IGV screenshot, so there is no plotting code to
recover. What is recovered is the data behind the junction track: the BED12
records, joined to the LeafCutter2 class and GENCODE annotation of each junction,
restricted to *SRSF4*. These are written to `source_data/` rather than plotted.

In [ ]:
for q, df in fig1c_tracks.items():
    counts = df.leafcutter2_category.value_counts().to_dict()
    print(f'{q:>4}: {len(df):3d} SRSF4 junctions  {counts}')

### Fig. 1d — classification by usage quartile

Junctions split into quartiles by usage, then the share of each LeafCutter2
class. Frequently used junctions are mostly productive; rarely used ones are
mostly not.

In [ ]:
P.plot_fig1d(fig1d_data)
P.save_panel('fig1d', PLOTS_DIR)

pct = fig1d_data.pivot(index='quartile', columns='category', values='pct_junctions').round(1)
print(pct.to_string())

### Fig. 1e — GENCODE composition of each class

What GENCODE v46 calls the junctions that LeafCutter2 assigns to each class.

In [ ]:
top = (fig1e_data.groupby('leafcutter2_category')
       .apply(lambda g: g.nlargest(4, 'n_junctions')[['gencode_annotation', 'pct_of_class']])
       )
print(top.round(1).to_string())

### Fig. 1g — log2 fold change, unproductive vs productive

Cumulative distribution of per-junction log2 fold change across four
perturbations that each stabilise NMD substrates. Unproductive junctions shift
right in every one; productive junctions sit on zero.

In [ ]:
P.plot_fig1g(fig1g_data)
P.save_panel('fig1g', PLOTS_DIR)

for s in fig1g_data:
    if s['category'] != 'utr':
        print(f"{s['comparison_label']:<36} {s['category_label']:<13} "
              f"n = {s['n']:>7,}   median log2FC = {s['median_log2fc']:+.3f}")

### Fig. 1h — Rules predicting NMD efficiency

A premature stop codon does not always trigger degradation. Each rule splits
unproductive junctions into the group it predicts is degraded more efficiently
and the group it predicts is degraded less, and compares their naRNA-vs-polyA
log2 fold change with a one-sided Mann-Whitney U test. Coding junctions, which
carry no premature stop, are the matched negative control.

The 50-nt rule is stated as a stop codon **further** than 50 nt from the last
exon-exon junction, which is the direction the rule predicts: a stop within
50 nt of the last junction escapes degradation.

In [ ]:
P.plot_fig1h(fig1h_data)
P.save_panel('fig1h', PLOTS_DIR)

cols = ['rule', 'class', 'n_low', 'n_high', 'mean_low', 'mean_high',
        'delta_log2fd', 'p_value']
print(fig1h_data[cols].to_string(index=False))